In [ ]:
import pandas as pd
import os
import json
import socket
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
model = 'neuchatel_2025'
with open('..//config.json', 'r') as file:
    mitgcm_config = json.load(file)[socket.gethostname()][model]

In [ ]:
start_date = '2025-08-01'
end_date = '2025-09-01'

In [ ]:
folder_path = os.path.dirname(mitgcm_config['datapath'])
output_folder = os.path.join(folder_path, "energy_budget")

In [ ]:
total_ke = pd.read_csv(os.path.join(output_folder, "ke_lake.csv"))

In [ ]:
eddy_ke = pd.read_csv(os.path.join(output_folder, "ke_eddies.csv"))

seiche_pe = pd.read_csv(os.path.join(output_folder, "potential_energy.csv"))

In [ ]:
seiche_ke = pd.read_csv(os.path.join(output_folder, "ke_seiche.csv"))

In [ ]:
mode1_ke = pd.read_csv(os.path.join(output_folder, "ke_mode1.csv"))

In [ ]:
wind_energy = pd.read_csv(os.path.join(output_folder, "E_wind_MJperh.csv"))

In [ ]:
total_ke['date'] = pd.to_datetime(total_ke['date'])
eddy_ke['date'] = pd.to_datetime(eddy_ke['date'])
#seiche_pe['date'] = pd.to_datetime(seiche_pe['time'])
seiche_ke['date'] = pd.to_datetime(seiche_ke['time'])
mode1_ke['date'] = pd.to_datetime(mode1_ke['time'])
wind_energy['date'] = pd.to_datetime(wind_energy['time'])

residual_ke['date'] = pd.to_datetime(residual_ke['time'])

In [ ]:
total_ke = total_ke.set_index('date')['kinetic_energy_[MJ]']
eddy_ke = eddy_ke.set_index('date')['kinetic_energy_eddy_[MJ]']
#seiche_pe = seiche_pe.set_index('date')['pe_mj_seiche']
seiche_ke = seiche_ke.set_index('date')['ke_mj_total']
mode1_ke = mode1_ke.set_index('date')['kinetic_energy_[MJ]']
wind_energy = wind_energy.set_index('date')['E_wind_MJperh']

In [ ]:
wind_energy_past_day = wind_energy.rolling(24).sum()

residual_ke = residual_ke.set_index('date')['ke_mj_seiche']

In [ ]:
plt.figure(figsize=(15,5))
total_ke.plot(label = 'Kinetic energy - Total')
seiche_ke.plot(label = 'Kinetic energy - Seiche')
mode1_ke.plot(label = 'Kinetic energy - EOF mode 1')
wind_energy_past_day.plot(label = 'Wind energy input')
#plt.gca().set_xlim(left=pd.to_datetime('2025-07-01'), right=pd.to_datetime('2025-07-05'))
plt.legend()

In [ ]:
plt.figure(figsize=(10,5))
total_ke.plot(label = 'Kinetic energy - Total')
eddy_ke.plot(label = 'Kinetic energy - Eddy')
seiche_ke.plot(label = 'Kinetic energy - Filtered 30-45h')
mode1_ke.plot(label = 'Kinetic energy - EOF mode 1')
wind_energy_past_day.plot(label = 'Wind energy input - last 24h', linestyle = '--')
#seiche_pe.plot(label = 'Potential energy', linestyle = '--')
#(seiche_pe + total_ke).plot(label = 'Total energy (kinetic + potential)', linestyle = ':')
#wind_energy.plot(label = 'Wind energy', color='black', linestyle = ':')
plt.title('Energy budget')
plt.ylabel('Energy [MJ]')
plt.gca().xaxis.set_major_locator(mdates.DayLocator())
plt.gca().set_xlim(left=pd.to_datetime(start_date), right=pd.to_datetime(end_date))
plt.xlabel('')
plt.xticks(rotation=90)
plt.legend()
plt.savefig(os.path.join(output_folder, 'budget.png'))

In [ ]:
plt.figure(figsize=(10,5))
(100*(eddy_ke+seiche_ke) / total_ke).plot(label = 'Eddy+Seiche', color='black')
(100*eddy_ke / total_ke).plot(label = 'Eddy', color='orange')
#(100*eof_mode1_ke / total_ke).plot(label = 'Seiche', color='green')
plt.title('Fraction energy budget')
plt.ylabel('Fraction [%]')
plt.gca().xaxis.set_major_locator(mdates.DayLocator())
plt.gca().set_xlim(left=pd.to_datetime(start_date), right=pd.to_datetime(end_date))
plt.gca().set_ylim(bottom=0, top=100)
plt.xlabel('')
plt.xticks(rotation=30)
plt.legend()
plt.savefig(os.path.join(output_folder, 'fraction.png'))

In [ ]:
plt.figure(figsize=(10,5))
(100*seiche_ke / total_ke).plot(label = 'Filtered KE', color='black')
(100*mode1_ke / total_ke).plot(label = 'EOF mode 1 KE', color='orange')
#(100*eof_mode1_ke / total_ke).plot(label = 'Seiche', color='green')
plt.title('Fraction energy budget')
plt.ylabel('Fraction [%]')
plt.gca().xaxis.set_major_locator(mdates.DayLocator())
plt.gca().set_xlim(left=pd.to_datetime(start_date), right=pd.to_datetime(end_date))
plt.gca().set_ylim(bottom=0, top=100)
plt.xlabel('')
plt.xticks(rotation=30)
plt.legend()
plt.savefig(os.path.join(output_folder, 'fraction.png'))